# 06 — OE5: Estabilidad por fases del ciclo del cobre

Tu **diferenciador**. Dos pasos:
1. **Fechar** el ciclo del cobre con el algoritmo BBQ (Harding-Pagan) — sobre el PRECIO DEL COBRE.
2. **Interactuar** la fase con los factores en el modelo OE2 -> ¿cambia la sensibilidad al cobre
   entre expansión y contracción? (asimetría de régimen, contrasta H1–H4 condicionadas).

Robustez opcional: Markov-switching (detección endógena de regímenes).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config as C
from src import cycle_dating as cd
from src import panel_models as pm

## 1. Fechado del ciclo del cobre (BBQ)

In [ ]:
# Cargar el precio del cobre mensual (de data/interim o processed)
macro = pd.read_parquet(C.DATA_INTERIM / 'raw_macro_global.parquet')
cobre_m = macro['cobre'].resample('ME').last().dropna()

fechado = cd.datar_ciclo_cobre(cobre_m, k=6, min_fase=6, min_ciclo=15)
fechado.loc[fechado['turning_point'].notna(), ['precio','turning_point']]

In [ ]:
# Visualizar fases sobre el precio del cobre
fig, ax = plt.subplots(figsize=(13,4))
ax.plot(fechado.index, fechado['precio'], color='black', lw=1)
exp = fechado['fase']=='expansion'
ax.fill_between(fechado.index, fechado['precio'].min(), fechado['precio'].max(),
                where=exp, alpha=.15, color='green', label='Expansión')
ax.fill_between(fechado.index, fechado['precio'].min(), fechado['precio'].max(),
                where=~exp, alpha=.15, color='red', label='Contracción')
picos = fechado['turning_point']=='P'; valles = fechado['turning_point']=='V'
ax.scatter(fechado.index[picos], fechado['precio'][picos], marker='v', color='red', zorder=5)
ax.scatter(fechado.index[valles], fechado['precio'][valles], marker='^', color='green', zorder=5)
ax.set_title('Fases del ciclo del precio del cobre (BBQ)'); ax.legend()
plt.tight_layout(); plt.savefig(C.FIGURES / 'oe5_fases_ciclo_cobre.png', dpi=120)

## 2. Modelo con interacción de fase (sensibilidad condicional)
$r_{i,t} = \alpha_i + \beta\,\Delta cobre_t + \delta\,(\Delta cobre_t \times exp_t) + \dots$

$\delta$ mide si la sensibilidad al cobre **cambia** en expansión vs contracción.

In [ ]:
panel = pd.read_parquet(C.DATA_PROCESSED / 'panel.parquet')
panel = cd.agregar_dummy_fase(panel, fechado)

# Términos de interacción con la dummy de expansión
for f in ['d_cobre','d_usdclp','d_tpm','d_vix']:
    if f in panel.columns:
        panel[f + '_x_exp'] = panel[f] * panel['exp']

regs = [c for c in ['d_cobre','d_usdclp','d_tpm','d_vix','exp',
                    'd_cobre_x_exp','d_usdclp_x_exp','d_tpm_x_exp','d_vix_x_exp']
        if c in panel.columns]
res_int = pm.panel_fe_driscoll_kraay(panel, dep='retorno', regresores=regs)
print(res_int)

## 3. Re-estimación por subperíodo (robustez)
Estima el modelo OE2 por separado en expansión y contracción y compara coeficientes.

In [ ]:
regs_base = [c for c in ['d_cobre','d_usdclp','d_tpm','d_vix'] if c in panel.columns]
resultados = {}
for fase in ['expansion','contraccion']:
    sub = panel[panel['fase']==fase]
    try:
        resultados[fase] = pm.panel_fe_driscoll_kraay(sub, 'retorno', regs_base).params
    except Exception as e:
        print(fase, 'error:', e)
comp = pd.DataFrame(resultados)
comp['dif'] = comp.get('expansion') - comp.get('contraccion')
display(comp)
comp.to_csv(C.TABLES / 'oe5_coef_por_fase.csv')

## 4. (Opcional) Markov-switching — detección endógena de régimen
Sobre la cartera del sector. Más exigente; usar solo como robustez.

In [ ]:
# from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
# cartera = panel['retorno'].groupby('fecha').mean().dropna()
# ms = MarkovRegression(cartera, k_regimes=2, switching_variance=True).fit()
# print(ms.summary())
# ms.smoothed_marginal_probabilities[0].plot(title='Prob. régimen 0')
print('Markov-switching opcional: descomentar si se requiere robustez de régimen.')

## 5. Lectura (OE5)
¿Es la sensibilidad al cobre mayor en expansión que en contracción? ¿Hay asimetría de
régimen? Concluye sobre la **estabilidad** de las relaciones a lo largo del ciclo.